# 02. 상태공간 모델

상태공간 표현은 로봇 시스템을 행렬로 정리하는 방식이다.

$$\dot{x}=Ax+Bu, \qquad y=Cx+Du$$

제어이론, Kalman Filter, LQR, MPC는 모두 이 표현을 기본 언어로 쓴다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 질량-스프링-댐퍼 시스템

상태 $x=[position, velocity]^T$ 로 두면:

$$\dot{x}=\begin{bmatrix}0&1\\-k/m&-c/m\end{bmatrix}x + \begin{bmatrix}0\\1/m\end{bmatrix}u$$

In [ ]:
m, c, k = 1.0, 0.45, 3.0
A = np.array([[0.0, 1.0], [-k/m, -c/m]])
B = np.array([[0.0], [1.0/m]])
C = np.array([[1.0, 0.0]])

print('A=')
print(A)
print('eigenvalues=', np.round(np.linalg.eigvals(A), 4))

def simulate(A, B, u_func, x0, dt=0.01, T=8.0):
    t = np.arange(0, T + dt, dt)
    x = np.zeros((len(t), len(x0)))
    x[0] = x0
    for i in range(len(t)-1):
        u = np.atleast_1d(u_func(t[i]))
        xdot = A @ x[i] + (B @ u).ravel()
        x[i+1] = x[i] + dt * xdot
    return t, x

t, x_free = simulate(A, B, lambda tt: [0.0], np.array([1.0, 0.0]))
t, x_step = simulate(A, B, lambda tt: [1.0 if tt > 0.5 else 0.0], np.array([0.0, 0.0]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(t, x_free[:,0], color='#534AB7', lw=2.5, label='position')
axes[0].plot(t, x_free[:,1], color='#E85D24', lw=2, label='velocity')
axes[0].set_title('자유 응답')
axes[0].grid(alpha=0.25); axes[0].legend()

axes[1].plot(t, x_step[:,0], color='#534AB7', lw=2.5, label='position')
axes[1].plot(t, x_step[:,1], color='#E85D24', lw=2, label='velocity')
axes[1].set_title('계단 입력 응답')
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout()
plt.savefig('assets/02_mass_spring_damper.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 위상평면(phase portrait)

상태공간에서 시스템이 어디로 흘러가는지 벡터장으로 확인한다.

In [ ]:
x1 = np.linspace(-2.0, 2.0, 21)
x2 = np.linspace(-3.0, 3.0, 21)
X1, X2 = np.meshgrid(x1, x2)
DX1 = X2
DX2 = -(k/m)*X1 - (c/m)*X2
N = np.sqrt(DX1**2 + DX2**2) + 1e-9

fig, ax = plt.subplots(figsize=(8, 7))
ax.quiver(X1, X2, DX1/N, DX2/N, color='gray', alpha=0.55)
for init, color in [(np.array([1.5, 0.0]), '#534AB7'), (np.array([-1.2, 2.0]), '#E85D24'), (np.array([0.4, -2.5]), '#1D9E75')]:
    _, traj = simulate(A, B, lambda tt: [0.0], init, dt=0.02, T=8.0)
    ax.plot(traj[:,0], traj[:,1], color=color, lw=2.5)
    ax.scatter(traj[0,0], traj[0,1], color=color, s=50)
ax.scatter(0, 0, color='black', s=60, label='equilibrium')
ax.set_xlabel('position'); ax.set_ylabel('velocity')
ax.set_title('위상평면: 감쇠 진동은 원점으로 수렴')
ax.grid(alpha=0.25); ax.legend()
plt.savefig('assets/02_phase_portrait.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 제어가능성(Controllability)

입력으로 모든 상태 방향을 움직일 수 있는지 확인한다.

$$\mathcal{C}=\begin{bmatrix}B & AB & A^2B & \cdots & A^{n-1}B\end{bmatrix}$$

rank가 상태 차원과 같으면 제어가능하다.

In [ ]:
def controllability_matrix(A, B):
    blocks = [B]
    for i in range(1, A.shape[0]):
        blocks.append(np.linalg.matrix_power(A, i) @ B)
    return np.hstack(blocks)

Ctrb = controllability_matrix(A, B)
print('Controllability matrix=')
print(np.round(Ctrb, 4))
print('rank=', np.linalg.matrix_rank(Ctrb), '/', A.shape[0])

# 비교: 두 상태가 서로 연결되지 않은 시스템에서 첫 번째 상태에만 입력이 들어가면
# 두 번째 상태는 제어할 수 없다.
A_unctrl = np.array([[-1.0, 0.0], [0.0, -2.0]])
B_unctrl = np.array([[1.0], [0.0]])
Ctrb_unctrl = controllability_matrix(A_unctrl, B_unctrl)
print('\nUncontrollable example=')
print(np.round(Ctrb_unctrl, 4))
print('rank=', np.linalg.matrix_rank(Ctrb_unctrl), '/', A_unctrl.shape[0])

## 요약

| 개념 | 수식 | 로보틱스 활용 |
|------|------|---------------|
| 상태공간 | $\dot{x}=Ax+Bu$ | 제어/추정의 공통 표현 |
| 고유값 | $A$의 동특성 | 안정성, 진동 모드 |
| 위상평면 | 상태 흐름 시각화 | 수렴/발산 직관 |
| 제어가능성 | $rank(\mathcal{C})=n$ | 입력으로 원하는 상태에 도달 가능한지 |